# MeterMind — Final Comparison

Loads results from all three models and produces a side-by-side comparison across all three metrics.

| Model | Description |
|-------|-------------|
| **DP Reorderer** | Rule-based, no learning. Finds optimal word ordering via dynamic programming |
| **GRU + Attention** | Learned seq2seq model trained on (prose, poem) pairs |
| **Claude (LLM)** | Large pretrained model prompted zero-shot |

**Metrics:**
- **MA** — Metrical Accuracy: fraction of syllables matching iambic template
- **SP** — Semantic Preservation: cosine similarity of sentence embeddings
- **G** — Grammaticality: normalised GPT-2 perplexity score

## 0. Install dependencies

In [ ]:
%pip install pandas matplotlib scipy --quiet

## 1. Imports

In [ ]:
import json, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from google.colab import files

print('Imports ready.')

## 2. Load results

Upload all three summary JSON files and all three results CSV files when prompted.

Files needed:
- `dp_reorderer_summary.json`
- `gru_attention_summary.json`
- `llm_summary.json`
- `dp_reorderer_results.csv`
- `gru_attention_results.csv`
- `llm_results.csv`

In [ ]:
print('Upload all 6 files (3 JSONs + 3 CSVs):')
uploaded = files.upload()

# Load summaries
summaries = []
for fname, data in uploaded.items():
    if fname.endswith('.json'):
        summaries.append(json.loads(data.decode('utf-8')))

# Load per-line results
result_dfs = {}
for fname, data in uploaded.items():
    if fname.endswith('.csv'):
        key = fname.replace('_results.csv', '')
        result_dfs[key] = pd.read_csv(io.StringIO(data.decode('utf-8')))

# Sort models consistently
MODEL_ORDER = ['DP Reorderer', 'GRU + Attention', 'Claude (LLM)']
summaries   = sorted(summaries, key=lambda x: MODEL_ORDER.index(x['model']) if x['model'] in MODEL_ORDER else 99)

print(f'\nLoaded {len(summaries)} model summaries and {len(result_dfs)} result DataFrames.')
for s in summaries:
    print(f"  {s['model']} — {s['n']} test pairs")

## 3. Summary table

In [ ]:
rows = []
for s in summaries:
    rows.append({
        'Model':   s['model'],
        'n':       s['n'],
        'MA mean': s['MA']['mean'],
        'MA std':  s['MA']['std'],
        'SP mean': s['SP']['mean'],
        'SP std':  s['SP']['std'],
        'G mean':  s['G']['mean'],
        'G std':   s['G']['std'],
    })

summary_df = pd.DataFrame(rows).set_index('Model')

# Highlight best per metric
def highlight_best(s):
    is_best = s == s.max()
    return ['font-weight: bold; color: green' if v else '' for v in is_best]

display(summary_df.style
    .apply(highlight_best, subset=['MA mean', 'SP mean', 'G mean'])
    .format(precision=4)
    .set_caption('MeterMind — Model Comparison (bold green = best)'))

## 4. Bar chart comparison

In [ ]:
metrics = ['MA', 'SP', 'G']
metric_labels = {
    'MA': 'Metrical Accuracy',
    'SP': 'Semantic Preservation',
    'G':  'Grammaticality'
}
colours = ['#4C72B0', '#55A868', '#C44E52']  # blue, green, red per model
models  = [s['model'] for s in summaries]
x       = np.arange(len(metrics))
width   = 0.25

fig, ax = plt.subplots(figsize=(10, 5))

for i, (s, colour) in enumerate(zip(summaries, colours)):
    means  = [s[m]['mean'] for m in metrics]
    stds   = [s[m]['std']  for m in metrics]
    offset = (i - 1) * width
    bars   = ax.bar(x + offset, means, width, yerr=stds,
                    label=s['model'], color=colour, alpha=0.85,
                    capsize=4, error_kw={'linewidth': 1.2})
    # Value labels on bars
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([metric_labels[m] for m in metrics], fontsize=11)
ax.set_ylabel('Score (mean ± std)', fontsize=11)
ax.set_title('MeterMind — Model Comparison', fontsize=13)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_barchart.png', dpi=150)
plt.show()

## 5. Distribution comparison (box plots)

In [ ]:
# Map CSV keys to model names
KEY_TO_MODEL = {
    'dp_reorderer':    'DP Reorderer',
    'gru_attention':   'GRU + Attention',
    'llm':             'Claude (LLM)',
}

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('MeterMind — Score Distributions', fontsize=13)

for ax, metric in zip(axes, metrics):
    data   = []
    labels = []
    for key, model_name in KEY_TO_MODEL.items():
        if key in result_dfs and metric in result_dfs[key].columns:
            data.append(result_dfs[key][metric].dropna().tolist())
            labels.append(model_name.replace(' + ', '\n+\n').replace(' (', '\n('))

    bp = ax.boxplot(data, labels=labels, patch_artist=True, notch=False)
    for patch, colour in zip(bp['boxes'], colours[:len(data)]):
        patch.set_facecolor(colour)
        patch.set_alpha(0.7)

    ax.set_title(metric_labels[metric], fontsize=11)
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1.05)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('comparison_boxplots.png', dpi=150)
plt.show()

## 6. Statistical significance testing

We use a **Wilcoxon signed-rank test** (non-parametric) to check whether differences between models are statistically significant.

This is more appropriate than a t-test here since metric distributions may not be normal.

In [ ]:
from itertools import combinations

key_list   = list(KEY_TO_MODEL.keys())
model_list = list(KEY_TO_MODEL.values())

print('=== Wilcoxon Signed-Rank Tests (p < 0.05 = significant) ===\n')

for metric in metrics:
    print(f'--- {metric_labels[metric]} ---')
    for (k1, k2) in combinations(key_list, 2):
        if k1 not in result_dfs or k2 not in result_dfs:
            continue
        if metric not in result_dfs[k1].columns or metric not in result_dfs[k2].columns:
            continue

        a = result_dfs[k1][metric].dropna().values
        b = result_dfs[k2][metric].dropna().values
        min_len = min(len(a), len(b))
        a, b = a[:min_len], b[:min_len]

        stat, p = stats.wilcoxon(a, b)
        sig = '✓ significant' if p < 0.05 else '✗ not significant'
        m1  = KEY_TO_MODEL[k1]
        m2  = KEY_TO_MODEL[k2]
        print(f'  {m1} vs {m2}: p={p:.4f}  {sig}')
    print()

## 7. Radar chart

A radar chart shows all three metrics at once for a quick visual summary.

In [ ]:
labels  = ['Metrical\nAccuracy', 'Semantic\nPreservation', 'Grammaticality']
n_metrics = len(labels)
angles  = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for s, colour in zip(summaries, colours):
    values = [s[m]['mean'] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, color=colour, label=s['model'])
    ax.fill(angles, values, alpha=0.15, color=colour)

ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=8)
ax.set_title('MeterMind — Radar Chart', fontsize=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('comparison_radar.png', dpi=150)
plt.show()

## 8. Qualitative examples

Side-by-side outputs from all three models on the same inputs.

In [ ]:
# Find inputs that appear in all three result sets for a fair comparison
if len(result_dfs) == 3:
    keys = list(result_dfs.keys())
    common_inputs = set(result_dfs[keys[0]]['input']) \
                  & set(result_dfs[keys[1]]['input']) \
                  & set(result_dfs[keys[2]]['input'])
    sample_inputs = list(common_inputs)[:5]

    print('=== Qualitative Examples ===\n')
    for inp in sample_inputs:
        print(f'INPUT  : {inp}')
        for key, model_name in KEY_TO_MODEL.items():
            if key not in result_dfs: continue
            row = result_dfs[key][result_dfs[key]['input'] == inp]
            if len(row) == 0: continue
            row = row.iloc[0]
            print(f'{model_name:20} : {row["output"]}  (MA={row["MA"]:.3f} SP={row["SP"]:.3f} G={row["G"]:.3f})')
        target = result_dfs[keys[0]][result_dfs[keys[0]]['input'] == inp].iloc[0]['target']
        print(f'TARGET               : {target}')
        print()
else:
    print('Need all 3 result CSVs uploaded for qualitative comparison.')

## 9. Save all outputs

In [ ]:
# Save summary table
summary_df.to_csv('metermind_comparison.csv')
print('Saved metermind_comparison.csv')

# Download all figures
for fname in ['comparison_barchart.png', 'comparison_boxplots.png', 'comparison_radar.png', 'metermind_comparison.csv']:
    try:
        files.download(fname)
        print(f'Downloaded {fname}')
    except:
        print(f'Could not download {fname} — may not have been generated yet')